# Assignment 5 — Option A: "Ask My Resume" RAG Chatbot
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Thomas Schlaerth  
**Date:** 13 May 2026  
**Option:** A — Resume RAG Chatbot  
**API Path:** Free  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Document Loading](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Retrieval Chain](#5-retrieval)
6. [Prompt Engineering](#6-prompting)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the RAG pipeline.  
> See the Option A Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `sentence-transformers` (free path)

In [97]:
# ── Install packages (uncomment as needed) ──
!pip install langchain langchain-community langchain-huggingface \
            sentence-transformers chromadb huggingface-hub \
            pypdf python-dotenv pandas langchain-text-splitters

# ── Load API keys from .env (or Colab Secrets) ──
import os
# If using a .env file locally:
# from dotenv import load_dotenv
# load_dotenv()

# If using Colab Secrets (recommended for Colab):
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN') # Access the token from Colab Secrets

# ── Your imports below ──
from langchain_community.document_loaders import PyPDFLoader, TextLoader # Corrected import
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter # Corrected import
from langchain_core.documents import Document # Corrected import
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from huggingface_hub import InferenceClient
import pandas as pd


# Clone your repo
if not os.path.exists('text-analytics-spring-2026'):
    !git clone https://github.com/Tschlaer/text-analytics-spring-2026.git

# Set the data directory to your specific folder
DATA_DIR = 'text-analytics-spring-2026/final-project/data'

print("Repo cloned")
print("Files in data folder:", os.listdir(DATA_DIR))

if HF_TOKEN:
    print('Hugging Face Token loaded successfully.')
else:
    print('Warning: Hugging Face Token not found. Please set it in Colab Secrets.')
print('Environment loaded')

Repo cloned
Files in data folder: ['Reference for Thomas Schlaerth.pdf', 'Thomas Schlaerth Cover Letter - Carson .pdf', 'DBS Cover Letter.pdf', 'Thomas_Schlaerth_Resume_2.16.26.pdf']
Hugging Face Token loaded successfully.
Environment loaded


---
<a id="2-loading"></a>
## 2. Document Loading

Load your 3-5 career documents from `data/`.  
Print: number of documents loaded, document types, and a sample of content to verify.

In [98]:
# ── Load your documents ──
DATA_DIR = 'text-analytics-spring-2026/final-project/data'

def load_documents(data_dir: str) -> list:
    """
    Load all .pdf and .txt documents from the data directory.
    Tags each document with its source filename in metadata.
    """
    docs = []
    files_seen = []

    for filename in sorted(os.listdir(data_dir)):
        filepath = os.path.join(data_dir, filename)

        if filename.endswith('.pdf'):
            loader = PyPDFLoader(filepath)
            loaded = loader.load()
            for doc in loaded:
                doc.metadata['source_file'] = filename
                doc.metadata['doc_type'] = 'PDF'
            docs.extend(loaded)
            files_seen.append(f'{filename} (PDF, {len(loaded)} page(s))')

        elif filename.endswith('.txt'):
            loader = TextLoader(filepath, encoding='utf-8')
            loaded = loader.load()
            for doc in loaded:
                doc.metadata['source_file'] = filename
                doc.metadata['doc_type'] = 'TXT'
            docs.extend(loaded)
            files_seen.append(f'{filename} (TXT, {len(loaded)} doc(s))')

    return docs, files_seen


documents, files_seen = load_documents(DATA_DIR)

print(f'Documents loaded: {len(files_seen)}')
for f in files_seen:
    print(f'  - {f}')

print(f'\nTotal pages/sections: {len(documents)}')
print('\n--- Sample content (first 400 chars of document 0) ---')
print(documents[0].page_content[:400])
print('\nMetadata:', documents[0].metadata)

Documents loaded: 4
  - DBS Cover Letter.pdf (PDF, 1 page(s))
  - Reference for Thomas Schlaerth.pdf (PDF, 1 page(s))
  - Thomas Schlaerth Cover Letter - Carson .pdf (PDF, 2 page(s))
  - Thomas_Schlaerth_Resume_2.16.26.pdf (PDF, 1 page(s))

Total pages/sections: 5

--- Sample content (first 400 chars of document 0) ---
1347  N  Euclid  Ave  
Tucson,  Arizona  85719    19  July  2022    Dear  Hiring  Manager,   I  hope  this  letter  finds  you  well.  I  am  writing  to  express  my  interest  in  the  Research  &  Data  Analyst  job  at  
your
 
company.
 
Over
 
this
 
past
 
summer,
 
I
 
have
 
participated
 
in
 
a
 
similar
 
role
 
in
 
Dublin,
 
Ireland.
 
From
 
what
 
I
 
learned
 
at
 
this
 
company


Metadata: {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'DBS Cover Letter', 'source': 'text-analytics-spring-2026/final-project/data/DBS Cover Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'DBS

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively (chunk count, avg length), and justify your final choice.

In [99]:
# ── Strategy 1: Fixed-size with overlap

splitter_fixed = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separator='\n'
)
chunks_fixed = splitter_fixed.split_documents(documents)

print('Strategy 1 — Fixed-size (CharacterTextSplitter)')
print(f'  Chunk count : {len(chunks_fixed)}')
lengths_fixed = [len(c.page_content) for c in chunks_fixed]
print(f'  Avg length  : {sum(lengths_fixed)/len(lengths_fixed):.0f} chars')
print(f'  Min / Max   : {min(lengths_fixed)} / {max(lengths_fixed)} chars')
print('\n--- Sample chunk (fixed) ---')
print(chunks_fixed[0].page_content)

Strategy 1 — Fixed-size (CharacterTextSplitter)
  Chunk count : 14
  Avg length  : 1063 chars
  Min / Max   : 123 / 3581 chars

--- Sample chunk (fixed) ---
1347  N  Euclid  Ave  
Tucson,  Arizona  85719    19  July  2022    Dear  Hiring  Manager,   I  hope  this  letter  finds  you  well.  I  am  writing  to  express  my  interest  in  the  Research  &  Data  Analyst  job  at  
your
 
company.
 
Over
 
this
 
past
 
summer,
 
I
 
have
 
participated
 
in
 
a
 
similar
 
role
 
in
 
Dublin,
 
Ireland.
 
From
 
what
 
I
 
learned
 
at
 
this
 
company
 
I
 
would
 
be
 
able
 
to
 
improve
 
your
 
company's
 
output
 
through
 
my
 
experience.
 
I


In [100]:
# ── Strategy 2: Recursive Character Text Splitter

splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ', '']
)
chunks_recursive = splitter_recursive.split_documents(documents)

print('Strategy 2 — Recursive (RecursiveCharacterTextSplitter)')
print(f'  Chunk count : {len(chunks_recursive)}')
lengths_rec = [len(c.page_content) for c in chunks_recursive]
print(f'  Avg length  : {sum(lengths_rec)/len(lengths_rec):.0f} chars')
print(f'  Min / Max   : {min(lengths_rec)} / {max(lengths_rec)} chars')
print('\n--- Sample chunk (recursive) ---')
print(chunks_recursive[0].page_content)

Strategy 2 — Recursive (RecursiveCharacterTextSplitter)
  Chunk count : 37
  Avg length  : 430 chars
  Min / Max   : 93 / 500 chars

--- Sample chunk (recursive) ---
1347  N  Euclid  Ave  
Tucson,  Arizona  85719    19  July  2022    Dear  Hiring  Manager,   I  hope  this  letter  finds  you  well.  I  am  writing  to  express  my  interest  in  the  Research  &  Data  Analyst  job  at  
your
 
company.
 
Over
 
this
 
past
 
summer,
 
I
 
have
 
participated
 
in
 
a
 
similar
 
role
 
in
 
Dublin,
 
Ireland.
 
From
 
what
 
I
 
learned
 
at
 
this
 
company
 
I
 
would
 
be
 
able
 
to
 
improve
 
your
 
company's
 
output
 
through
 
my
 
experience.
 
I


In [101]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    'Strategy': ['Fixed-size (CharacterTextSplitter)', 'Recursive (RecursiveCharacterTextSplitter)'],
    'Chunk Count': [len(chunks_fixed), len(chunks_recursive)],
    'Avg Length (chars)': [
        round(sum(lengths_fixed)/len(lengths_fixed)),
        round(sum(lengths_rec)/len(lengths_rec))
    ],
    'Min Length': [min(lengths_fixed), min(lengths_rec)],
    'Max Length': [max(lengths_fixed), max(lengths_rec)]
})
print(comparison.to_string(index=False))

                                  Strategy  Chunk Count  Avg Length (chars)  Min Length  Max Length
        Fixed-size (CharacterTextSplitter)           14                1063         123        3581
Recursive (RecursiveCharacterTextSplitter)           37                 430          93         500


### Chunking Decision

**Which strategy did you choose?** I decided to use the Recursive Character Text Splitter because the average characters are much smaller than the Fixed-Size method.

**Why?**  The Recursive method is more applicable with my document choices. I decided to add my resume, cover letters, and a letter of recommendation. These documents are formatted in paragraphs and sentences which perform better than fixed-size. If I was to choose fixed-size there would be a possiblity that the sentences would be cut off midway through.

**Final settings (chunk_size, overlap):** chunk_size=500 and chunk_overlap=50
    

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [102]:
# ── Create embeddings and vector store ──

CHROMA_PATH = '../chroma_db'
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# Use the recursive chunks selected in Section 3
chunks = chunks_recursive

print(f'Loading embedding model: {EMBEDDING_MODEL} ...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'}
)
print('Embedding model loaded')

print(f'\nEmbedding {len(chunks)} chunks into ChromaDB...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)
vectorstore.persist()

print(f'Vector store saved to {CHROMA_PATH}')
print(f'Total vectors stored: {vectorstore._collection.count()}')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded

Embedding 37 chunks into ChromaDB...
Vector store saved to ../chroma_db
Total vectors stored: 222


In [103]:
# ── Verify: run a test similarity search ──
def test_search(query: str, k: int = 3) -> None:
    """Run a similarity search and display the top-k retrieved chunks."""
    print(f'Query: "{query}"')
    print('-' * 60)
    results = vectorstore.similarity_search(query, k=k)
    for i, doc in enumerate(results):
        source = doc.metadata.get('source_file', 'unknown')
        print(f'[Result {i+1}] Source: {source}')
        print(doc.page_content[:300])
        print()

# Test 1 — Technical skills
test_search('Python programming skills')

# Test 2 — Work experience
test_search('most recent job or internship experience')

# Test 3 — Education
test_search('university degree and coursework')

Query: "Python programming skills"
------------------------------------------------------------
[Result 1] Source: Thomas_Schlaerth_Resume_2.16.26.pdf
of Los Angeles (2019) § Most Improved Racer – Cycling Team, Loyola High School of Los Angeles (2017)   SKILLS & ABILITIES § Languages: Bilingual English and Hungarian; Novice Proficiency in Chinese/Mandarin and Spanish § Skills: Proficient in Microsoft Office Suite (Word, Excel, Access, PowerPoint, 

[Result 2] Source: Thomas_Schlaerth_Resume_2.16.26.pdf
of Los Angeles (2019) § Most Improved Racer – Cycling Team, Loyola High School of Los Angeles (2017)   SKILLS & ABILITIES § Languages: Bilingual English and Hungarian; Novice Proficiency in Chinese/Mandarin and Spanish § Skills: Proficient in Microsoft Office Suite (Word, Excel, Access, PowerPoint, 

[Result 3] Source: Thomas_Schlaerth_Resume_2.16.26.pdf
of Los Angeles (2019) § Most Improved Racer – Cycling Team, Loyola High School of Los Angeles (2017)   SKILLS & ABILITIES § Languages: 

---
<a id="5-retrieval"></a>
## 5. Retrieval Chain

Connect your vector store to an LLM to build the full RAG chain.

**Paid path:** OpenAI `gpt-4o-mini` or `gpt-3.5-turbo`  
**Free path:** HuggingFace Inference API or Ollama

In [104]:
# ── Initialize LLM ──
from huggingface_hub import InferenceClient

HF_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

client = InferenceClient(
    model=HF_MODEL,
    token=HF_TOKEN
)
print(f'LLM client ready: {HF_MODEL}')

LLM client ready: Qwen/Qwen2.5-7B-Instruct


In [105]:
# ── Build retrieval chain ──
def ask(question: str, system_prompt: str, k: int = 3) -> dict:
    """
    Full RAG pipeline:
      1. Retrieve top-k relevant chunks from ChromaDB
      2. Inject context into system prompt
      3. Call LLM via HuggingFace Inference API
      4. Return answer + retrieved chunks

    Args:
        question (str): The user's natural language question
        system_prompt (str): System prompt template containing {context} placeholder
        k (int): Number of chunks to retrieve (default 3)

    Returns:
        dict with keys 'question', 'answer', 'chunks'
    """
    # Step 1: Retrieve
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    retrieved = retriever.invoke(question)
    context = '\n\n---\n\n'.join([c.page_content for c in retrieved])

    # Step 2: Build prompt
    filled_prompt = system_prompt.replace('{context}', context)

    # Step 3: Generate
    response = client.chat_completion(
        messages=[
            {'role': 'system', 'content': filled_prompt},
            {'role': 'user',   'content': question}
        ],
        max_tokens=512,
        temperature=0.2
    )
    answer = response.choices[0].message.content.strip()

    return {'question': question, 'answer': answer, 'chunks': retrieved}


# Smoke test
SYSTEM_PROMPT_PLACEHOLDER = (
    'You are a helpful assistant. Answer questions about this person using the context below.\n\n'
    'Context:\n{context}'
)
test = ask('What is this person\'s educational background?', SYSTEM_PROMPT_PLACEHOLDER)
print('Q:', test['question'])
print('A:', test['answer'])
print(f'\nChunks used: {len(test["chunks"])}')

Q: What is this person's educational background?
A: The provided context does not include any information about the person's educational background. It only describes their volunteer tutoring experience at Magnolia Elementary School in Los Angeles from September 2016 to June 2020. To determine their educational background, additional information would be needed.

Chunks used: 3


---
<a id="6-prompting"></a>
## 6. Prompt Engineering

Design your system prompt for the RAG chain. Consider: grounding (answer only from context), tone, how to handle out-of-scope questions, response format.

**Required:** Show at least 3 iterations. For each, explain what you changed, why, and show a before/after test with the same question.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [106]:
# ── Prompt Version 1: Basic instruction ──

PROMPT_V1 = (
    'You are a helpful assistant. '
    'Answer questions about this person\'s resume.\n\n'
    'Context:\n{context}'
)

# ── Prompt Version 2: Added grounding constraint + fallback phrase ──

PROMPT_V2 = (
    'You are a professional assistant representing a job candidate.\n'
    'Answer questions using ONLY the information in the context provided below.\n'
    'Do not add information that is not in the context.\n'
    'If the context does not contain the answer, say exactly: '
    '"That information is not available in the candidate\'s documents."\n\n'
    'Context:\n{context}'
)

# ── Prompt Version 3: Added tone, format rules, sensitive topic guard ──

PROMPT_V3 = (
    'You are a professional assistant representing a job candidate to recruiters and hiring managers.\n\n'
    'INSTRUCTIONS:\n'
    '- Answer using ONLY the information provided in the context below.\n'
    '- Do not invent, assume, or extrapolate facts not in the context.\n'
    '- If the context does not contain enough information, respond with:\n'
    '  "That information is not available in the candidate\'s documents."\n'
    '- Write in third-person professional tone (e.g., "The candidate has...").\n'
    '- Cite the specific document or project when relevant.\n'
    '- Do not discuss salary, compensation, or personal contact information.\n\n'
    'Context:\n{context}'
)

print('Prompts v1, v2, v3 defined')

Prompts v1, v2, v3 defined


In [107]:
# ── Before/After comparison: same question across all 3 versions ──

TEST_Q = "What industry is this person trying to enter?"

for label, prompt in [
    ('v1 — Basic', PROMPT_V1),
    ('v2 — Grounding added', PROMPT_V2),
    ('v3 — Full (Final)', PROMPT_V3)
]:
    result = ask(TEST_Q, prompt)
    print(f'=== {label} ===')
    print('A:', result['answer'])
    print()

=== v1 — Basic ===
A: The person is trying to enter the logistics industry. Specifically, they are interested in roles within logistics management, as evidenced by their application for a Logistics Trainee position at Penske Logistics.

=== v2 — Grounding added ===
A: The person is trying to enter the logistics industry.

=== v3 — Full (Final) ===
A: The candidate is trying to enter the logistics industry.



In [108]:
# ── Also compare on a factual question to see tone improvement ──

TEST_Q2 = "What technical skills does this person have?"

for label, prompt in [
    ('v2 — Grounding added', PROMPT_V2),
    ('v3 — Full (Final)', PROMPT_V3)
]:
    result = ask(TEST_Q2, prompt)
    print(f'=== {label} ===')
    print('A:', result['answer'])
    print()

# Set final prompt for use in evaluation
SYSTEM_PROMPT = PROMPT_V3
print('Final prompt set to v3')

=== v2 — Grounding added ===
A: This person has the following technical skills:
- Proficient in Microsoft Office Suite (Word, Excel, Access, PowerPoint, Visio)
- Knowledge of Python 3 Computer Programming
- Knowledge in Tableau, Figma, and Jira
- Proficient in Computer Hardware Assembly and Repair

=== v3 — Full (Final) ===
A: The candidate has proficiency in the Microsoft Office Suite (Word, Excel, Access, PowerPoint, Visio), knowledge of Python 3 computer programming, and proficiency in tools such as Tableau, Figma, and Jira. Additionally, the candidate is proficient in computer hardware assembly and repair.

Final prompt set to v3


---
<a id="7-evaluation"></a>
## 7. Evaluation

Test your chatbot with **10 questions** across 4 categories:
- Factual retrieval (2-3): questions with clear answers in your docs
- Inference (2-3): questions requiring reasoning across your docs
- Out-of-scope (2-3): questions your docs cannot answer
- Specificity (2-3): questions targeting a specific document

For each question, score: **retrieval quality** (Yes/Partial/No), **faithfulness** (Faithful/Partial/Hallucinated), **answer quality** (1-5).

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [109]:
# ── Define your 10 test questions and run evaluation ──

TEST_QUESTIONS = [
    # Factual retrieval
    ('Factual', 'What programming languages does this person know?'),
    ('Factual', 'What is this person\'s educational background?'),
    ('Factual', 'What tools or software does this person have experience with?'),
    # Inference
    ('Inference', 'Would this person be a good fit for a data engineering role?'),
    ('Inference', 'Does this person have leadership or team experience?'),
    ('Inference', 'What industries or domains has this person worked in?'),
    # Out-of-scope
    ('Out-of-scope', 'What is this person\'s salary expectation?'),
    ('Out-of-scope', 'What is this person\'s home address?'),
    # Specificity
    ('Specificity', 'Describe the most recent project this person worked on.'),
    ('Specificity', 'What were this person\'s responsibilities in their last role?'),
]

print('Running 10 evaluation queries...\n')
raw_results = []

for category, question in TEST_QUESTIONS:
    result = ask(question, SYSTEM_PROMPT, k=3)
    raw_results.append({
        'category': category,
        'question': question,
        'answer': result['answer'],
        'n_chunks': len(result['chunks']),
        'sources': [c.metadata.get('source_file', 'unknown') for c in result['chunks']]
    })
    print(f'[{category}] {question}')
    print(f'  A: {result["answer"][:200]}...' if len(result['answer']) > 200 else f'  A: {result["answer"]}')
    print(f'  Sources: {set(raw_results[-1]["sources"])}')
    print()

Running 10 evaluation queries...

[Factual] What programming languages does this person know?
  A: That information is not available in the candidate's documents.
  Sources: {'Thomas Schlaerth Cover Letter - Carson .pdf'}

[Factual] What is this person's educational background?
  A: That information is not available in the candidate's documents.
  Sources: {'Thomas_Schlaerth_Resume_2.16.26.pdf'}

[Factual] What tools or software does this person have experience with?
  A: That information is not available in the candidate's documents.
  Sources: {'DBS Cover Letter.pdf'}

[Inference] Would this person be a good fit for a data engineering role?
  A: That information is not available in the candidate's documents.
  Sources: {'DBS Cover Letter.pdf'}

[Inference] Does this person have leadership or team experience?
  A: The candidate has developed leadership through diverse team roles, indicating experience in leading teams.
  Sources: {'Thomas_Schlaerth_Resume_2.16.26.pdf'}

[Inference] Wh

In [110]:
import os

# After running the queries above, fill in your manual ratings in the lists below.
# Retrieval Quality  : 'Yes' / 'Partial' / 'No'
# Answer Faithfulness: 'Faithful' / 'Partial' / 'Hallucinated'
# Answer Quality     : 1 (poor) to 5 (excellent)

eval_data = {
    'Category':            [r['category'] for r in raw_results],
    'Question':            [r['question'] for r in raw_results],
    'Retrieval Quality':   ['1', '1', '1', '4', '5', '1', '5', '5', '3', '2'],   # ← fill in
    'Answer Faithfulness': ['4', '5', '3', '5', '5', '1', '5', '5', '4', '4'],   # ← fill in
    'Answer Quality (1-5)':['1', '1', '1', '5', '5', '1', '5', '5', '3', '3'],   # ← fill in
    'Notes':               ['Wrong Doc', 'Not retrieved from doc', 'Wrong Doc', 'Good answer using context', 'Good answer using context', 'Not retrieved from doc', 'Good Answer', 'Good Answer', 'Not most recent project', 'Not last role'],   # ← optional
}

eval_df = pd.DataFrame(eval_data)
print(eval_df.to_string(index=False))

# Save to current folder
output_dir = '.'
os.makedirs(output_dir, exist_ok=True)
eval_df.to_csv(os.path.join(output_dir, 'test_results.csv'), index=False)
print('Saved to test_results.csv in the current directory')

    Category                                                      Question Retrieval Quality Answer Faithfulness Answer Quality (1-5)                     Notes
     Factual             What programming languages does this person know?                 1                   4                    1                 Wrong Doc
     Factual                 What is this person's educational background?                 1                   5                    1    Not retrieved from doc
     Factual What tools or software does this person have experience with?                 1                   3                    1                 Wrong Doc
   Inference  Would this person be a good fit for a data engineering role?                 4                   5                    5 Good answer using context
   Inference          Does this person have leadership or team experience?                 5                   5                    5 Good answer using context
   Inference         What industries or 

### Evaluation Analysis

**Where does the chatbot succeed?** Gathering specific skills, software, technical details from certain documents.

**Where does it fail? Why?** Choosing incorrect documents to answer questions. Some questions had answers listed in a different document which the model did not choose.

**What would you improve?** Making sure the model checks all documents before answering a question. Making sure  necessary content is taken from any/all documents.

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option A*